# 09 — ML Models

Predictive models supporting the research questions:
- **LOS regression** (RQ2, RQ4): predict length of stay from patient and hospital features
- **Long-stay classification** (RQ4, RQ5): predict which patients will stay >7 days
- **SHAP interpretation** (RQ2, RQ6): which features drive LOS and long-stay risk

These models are analytical tools — they feed insights into the RQ notebooks, not standalone deliverables.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, r2_score, mean_absolute_error
import lightgbm as lgb

from shared import (
    load_kidney, load_hospital_tags, load_cnes_enriched,
    setup_plot_style, save_plot, save_metrics, DATA_DIR,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
cnes = load_cnes_enriched()
print(f"Loaded {len(kidney):,} admissions")

Loaded 206,500 admissions


## Feature Engineering

In [2]:
# Patient-level features from SIH
features = kidney[["CNES", "DIAS_PERM", "age", "is_male", "is_emergency",
                    "has_new_proc", "proc_category", "migrated", "MORTE"]].copy()

# Encode procedure category
proc_dummies = pd.get_dummies(features["proc_category"], prefix="proc", drop_first=True)
features = pd.concat([features, proc_dummies], axis=1)

# Hospital-level features from tags
hosp_features = tags[["CNES", "n_admissions", "elective_rate", "mortality_rate",
                       "facility_type", "admission_profile", "case_mix_profile"]].copy()

# Encode hospital categoricals
for col in ["facility_type", "admission_profile", "case_mix_profile"]:
    dummies = pd.get_dummies(hosp_features[col], prefix=col, drop_first=True)
    hosp_features = pd.concat([hosp_features, dummies], axis=1)
hosp_features = hosp_features.drop(columns=["facility_type", "admission_profile", "case_mix_profile"])

features = features.merge(hosp_features, on="CNES", how="left")

# CNES enriched features
cnes_feats = ["CNES", "total_beds", "surgical_beds", "icu_beds",
              "doctors_per_bed", "nurses_per_bed", "total_equipment",
              "active_committees"]
available_cnes = [c for c in cnes_feats if c in cnes.columns]
if len(available_cnes) > 1:
    cnes_subset = cnes[available_cnes].drop_duplicates(subset=["CNES"])
    features = features.merge(cnes_subset, on="CNES", how="left")

# Target variables
features["is_long_stay"] = (features["DIAS_PERM"] > 7).astype(int)

# Drop non-feature columns
drop_cols = ["CNES", "DIAS_PERM", "proc_category", "MORTE", "is_long_stay"]
feature_cols = [c for c in features.columns if c not in drop_cols]

X = features[feature_cols].fillna(0).astype(float)
y_los = features["DIAS_PERM"]
y_long = features["is_long_stay"]

print(f"Feature matrix: {X.shape}")
print(f"Features: {list(X.columns)}")

Feature matrix: (206500, 32)
Features: ['age', 'is_male', 'is_emergency', 'has_new_proc', 'migrated', 'proc_DIAGNOSTIC', 'proc_INTERVENTIONAL', 'proc_OBSERVATION', 'proc_OTHER', 'proc_SURGICAL', 'proc_SURGICAL_MGMT', 'n_admissions', 'elective_rate', 'mortality_rate', 'facility_type_hospital_especializado', 'facility_type_hospital_geral', 'facility_type_other', 'facility_type_pronto_socorro', 'facility_type_unidade_mista', 'facility_type_unknown', 'admission_profile_emergency_dominant', 'admission_profile_mixed', 'case_mix_profile_diagnostic_heavy', 'case_mix_profile_mixed_procedures', 'case_mix_profile_surgical_center', 'total_beds', 'surgical_beds', 'icu_beds', 'doctors_per_bed', 'nurses_per_bed', 'total_equipment', 'active_committees']


## 1. LOS Regression

In [3]:
reg_model = lgb.LGBMRegressor(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, verbose=-1, n_jobs=-1,
)

cv_r2 = cross_val_score(reg_model, X, y_los, cv=5, scoring="r2")
cv_mae = -cross_val_score(reg_model, X, y_los, cv=5, scoring="neg_mean_absolute_error")

print(f"LOS Regression (5-fold CV):")
print(f"  R²: {cv_r2.mean():.3f} ± {cv_r2.std():.3f}")
print(f"  MAE: {cv_mae.mean():.2f} ± {cv_mae.std():.2f} days")

# Fit on full data for feature importance
reg_model.fit(X, y_los)
reg_importance = pd.Series(reg_model.feature_importances_, index=X.columns).sort_values(ascending=True)

LOS Regression (5-fold CV):
  R²: 0.155 ± 0.024
  MAE: 1.49 ± 0.08 days


## 2. Long-Stay Classification

In [4]:
clf_model = lgb.LGBMClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1,
    random_state=42, verbose=-1, n_jobs=-1,
    scale_pos_weight=len(y_long[y_long == 0]) / max(len(y_long[y_long == 1]), 1),
)

cv_auc = cross_val_score(clf_model, X, y_long, cv=StratifiedKFold(5), scoring="roc_auc")

print(f"Long-Stay Classification (5-fold CV):")
print(f"  AUC: {cv_auc.mean():.3f} ± {cv_auc.std():.3f}")

# Fit on full data for feature importance
clf_model.fit(X, y_long)
clf_importance = pd.Series(clf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

Long-Stay Classification (5-fold CV):
  AUC: 0.781 ± 0.016


## 3. Feature Importance Comparison

In [5]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

reg_importance.tail(15).plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title(f"LOS Regression\n(R²={cv_r2.mean():.3f})")
axes[0].set_xlabel("Feature Importance")

clf_importance.tail(15).plot.barh(ax=axes[1], color="#DC2626")
axes[1].set_title(f"Long-Stay Classification\n(AUC={cv_auc.mean():.3f})")
axes[1].set_xlabel("Feature Importance")

fig.suptitle("Feature Importance — LOS Regression vs Long-Stay Classification",
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "feature_importance", prefix="09")

  Saved: 09_feature_importance.png


## 4. SHAP Analysis

In [6]:
try:
    import shap

    # Sample for SHAP computation speed
    sample_idx = np.random.RandomState(42).choice(len(X), size=min(5000, len(X)), replace=False)
    X_sample = X.iloc[sample_idx]

    # Regression SHAP
    explainer_reg = shap.TreeExplainer(reg_model)
    shap_reg = explainer_reg.shap_values(X_sample)

    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_reg, X_sample, show=False, max_display=15)
    plt.title("SHAP — LOS Regression", fontsize=14)
    plt.tight_layout()
    save_plot(plt.gcf(), "shap_regression", prefix="09")

    # Classification SHAP
    explainer_clf = shap.TreeExplainer(clf_model)
    shap_clf = explainer_clf.shap_values(X_sample)
    if isinstance(shap_clf, list):
        shap_clf = shap_clf[1]  # positive class

    fig, ax = plt.subplots(figsize=(12, 8))
    shap.summary_plot(shap_clf, X_sample, show=False, max_display=15)
    plt.title("SHAP — Long-Stay Classification", fontsize=14)
    plt.tight_layout()
    save_plot(plt.gcf(), "shap_longstay", prefix="09")

    print("SHAP plots generated.")
except ImportError:
    print("SHAP not installed. Skipping SHAP analysis.")
except Exception as e:
    print(f"SHAP analysis failed: {e}")

/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Saved: 09_shap_regression.png


/Users/gsampaio/redhat/customers/prodesp/health-sus-agent/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


  Saved: 09_shap_longstay.png
SHAP plots generated.


## Summary Metrics

In [7]:
metrics = {
    "n_features": int(X.shape[1]),
    "n_samples": int(X.shape[0]),
    "regression_r2_mean": round(float(cv_r2.mean()), 3),
    "regression_r2_std": round(float(cv_r2.std()), 3),
    "regression_mae_mean": round(float(cv_mae.mean()), 2),
    "classification_auc_mean": round(float(cv_auc.mean()), 3),
    "classification_auc_std": round(float(cv_auc.std()), 3),
    "long_stay_prevalence": round(float(y_long.mean() * 100), 1),
    "top_reg_feature": str(reg_importance.index[-1]),
    "top_clf_feature": str(clf_importance.index[-1]),
}
save_metrics(metrics, "09_ml_models")

print("\n=== ML MODELS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 09_ml_models.json

=== ML MODELS ===
  n_features: 32
  n_samples: 206500
  regression_r2_mean: 0.155
  regression_r2_std: 0.024
  regression_mae_mean: 1.49
  classification_auc_mean: 0.781
  classification_auc_std: 0.016
  long_stay_prevalence: 4.5
  top_reg_feature: age
  top_clf_feature: age
